In [53]:
import pandas as pd
import geopandas as gpd
import tobler   
import numpy as np

In [80]:
pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', lambda x: '%.15f' % x)  # Exibe 5 casas decimais


In [ ]:
sectors_2022 = gpd.read_file('/Volumes/ssd/Doutorado/DATASET/IBGE/Setores Censitários/setores_censitarios_2022/br/setores_censitarios_pop_fcu.shp')
sectors_2022.set_crs("EPSG:4674", inplace=True)
sectors_2022.columns = ['CD_GEOCOD_2022', 'pop_2022', 'CD_FCU_2022', 'geometry']
sectors_2022

,CD_GEOCOD_2022,pop_2022,CD_FCU_2022,geometry
0,120001305000001,920,None,"POLYGON ((-67.04925 -10.07069, -67.04938 -10.0..."
1,120001305000002,354,None,"POLYGON ((-67.03937 -10.07846, -67.03897 -10.0..."
2,120001305000005,278,None,"POLYGON ((-67.03598 -10.00512, -67.03653 -10.0..."
3,120001305000006,401,None,"POLYGON ((-66.91526 -9.86712, -66.91544 -9.867..."
4,120001305000007,241,None,"POLYGON ((-66.88699 -9.83206, -66.88808 -9.832..."
...,...,...,...,...
489073,172210705000030,276,17221070001,"POLYGON ((-48.53139 -6.41658, -48.53136 -6.416..."
489074,172210705000031,289,None,"POLYGON ((-48.52973 -6.41638, -48.52999 -6.416..."
489075,172210705000032,152,None,"POLYGON ((-48.53265 -6.41567, -48.53287 -6.415..."
489076,172210705000034,393,None,"POLYGON ((-48.53492 -6.41734, -48.53513 -6.417..."


In [ ]:
sectors_2010 = gpd.read_file('/Volumes/ssd/Doutorado/DATASET/IBGE/Setores Censitários/setores_censitarios_2010/br/setores_censitarios_pop_ibp.shp')
sectors_2010.set_crs("EPSG:4674", inplace=True)
sectors_2010.columns = ['ID', 'CD_GEOCOD_2010', 'pop_2010','IBP_2010', 'geometry']
sectors_2010

,ID,CD_GEOCOD_2010,pop_2010,IBP_2010,geometry
0,17182,110009812000003,555.0,3.128592,"POLYGON ((-60.89575 -11.35508, -60.89557 -11.3..."
1,17183,110009815000001,53.0,2.838014,"POLYGON ((-60.74999 -11.3999, -60.74913 -11.40..."
2,17184,110009815000002,507.0,2.903042,"POLYGON ((-60.72986 -11.35738, -60.72954 -11.3..."
3,17185,110009815000003,555.0,3.543457,"POLYGON ((-60.91829 -11.29374, -60.916 -11.292..."
4,17186,110009815000004,336.0,3.319936,"POLYGON ((-60.69047 -11.38391, -60.6903 -11.38..."
...,...,...,...,...,...
316569,4450,530010805300153,434.0,-2.459134,"POLYGON ((-47.81165 -15.86005, -47.80981 -15.8..."
316570,4451,530010805300154,534.0,-3.100082,"POLYGON ((-47.81951 -15.8618, -47.81866 -15.85..."
316571,4452,530010805300155,532.0,-2.973202,"POLYGON ((-47.81758 -15.85556, -47.81269 -15.8..."
316572,4453,530010805300156,2103.0,-0.975027,"POLYGON ((-47.7841 -15.90137, -47.78145 -15.90..."


# Dasymetric Interpolation - Pop 2022 ancillary

In [ ]:
# Criar interseção entre setores de 2010 e 2022
intersection = gpd.overlay(sectors_2010, sectors_2022, how="intersection")
intersection

/var/folders/5l/x3ysgz0n0mvd7msxlmn75zhw0000gp/T/ipykernel_8429/1949481993.py:2: UserWarning: `keep_geom_type=True` in overlay resulted in 30034 dropped geometries of different geometry types than df1 has. Set `keep_geom_type=False` to retain all geometries
  intersection = gpd.overlay(setores_2010, setores_2022, how="intersection")


,ID,CD_GEOCOD_2010,pop_2010,IBP_2010,CD_GEOCOD_2022,pop_2022,CD_FCU_2022,geometry
0,17182,110009812000003,555.0,3.128592,110009805000022,616,None,"MULTIPOLYGON (((-60.83109 -11.48749, -60.83352..."
1,17182,110009812000003,555.0,3.128592,110009805000083,495,None,"POLYGON ((-60.99222 -11.5001, -60.9796 -11.483..."
2,17182,110009812000003,555.0,3.128592,110009812000001,173,None,"MULTIPOLYGON (((-60.92784 -11.4616, -60.92784 ..."
3,17182,110009812000003,555.0,3.128592,110009812000002,714,None,"MULTIPOLYGON (((-60.98469 -11.47198, -60.96361..."
4,17182,110009812000003,555.0,3.128592,110009812000003,391,None,"POLYGON ((-60.89413 -11.35596, -60.89401 -11.3..."
...,...,...,...,...,...,...,...,...
2179385,4454,530010805300158,169.0,NaN,530010805300273,221,None,"POLYGON ((-47.7822 -15.91286, -47.7822 -15.912..."
2179386,4454,530010805300158,169.0,NaN,530010805300274,206,None,"POLYGON ((-47.7861 -15.92517, -47.7862 -15.925..."
2179387,4454,530010805300158,169.0,NaN,530010805400043,3586,None,"MULTIPOLYGON (((-47.78323 -15.92072, -47.78462..."
2179388,4454,530010805300158,169.0,NaN,530010805400082,0,None,"POLYGON ((-47.78991 -15.91907, -47.78883 -15.9..."


In [ ]:
# Calcular a área da interseção
intersection["area_intersection"] = intersection.geometry.area
sectors_2022["area_total"] = sectors_2022.geometry.area


# Mesclar os dados de IBP e população de 2010
intersection = intersection.merge(sectors_2022[["CD_GEOCOD_2022", "area_total"]], on="CD_GEOCOD_2022")
intersection

/var/folders/5l/x3ysgz0n0mvd7msxlmn75zhw0000gp/T/ipykernel_8429/2639440563.py:2: UserWarning: Geometry is in a geographic CRS. Results from 'area' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  intersection["area_intersection"] = intersection.geometry.area
/var/folders/5l/x3ysgz0n0mvd7msxlmn75zhw0000gp/T/ipykernel_8429/2639440563.py:3: UserWarning: Geometry is in a geographic CRS. Results from 'area' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  setores_2022["area_total"] = setores_2022.geometry.area


,ID,CD_GEOCOD_2010,pop_2010,IBP_2010,CD_GEOCOD_2022,pop_2022,CD_FCU_2022,geometry,area_intersection,area_total
0,17182,110009812000003,555.0,3.128592,110009805000022,616,None,"MULTIPOLYGON (((-60.83109 -11.48749, -60.83352...",1.343150e-04,0.016273
1,17182,110009812000003,555.0,3.128592,110009805000083,495,None,"POLYGON ((-60.99222 -11.5001, -60.9796 -11.483...",2.304771e-05,0.003834
2,17182,110009812000003,555.0,3.128592,110009812000001,173,None,"MULTIPOLYGON (((-60.92784 -11.4616, -60.92784 ...",2.824522e-10,0.000033
3,17182,110009812000003,555.0,3.128592,110009812000002,714,None,"MULTIPOLYGON (((-60.98469 -11.47198, -60.96361...",1.694423e-04,0.018971
4,17182,110009812000003,555.0,3.128592,110009812000003,391,None,"POLYGON ((-60.89413 -11.35596, -60.89401 -11.3...",9.471774e-03,0.009793
...,...,...,...,...,...,...,...,...,...,...
2365033,4454,530010805300158,169.0,NaN,530010805300273,221,None,"POLYGON ((-47.7822 -15.91286, -47.7822 -15.912...",9.990698e-15,0.000040
2365034,4454,530010805300158,169.0,NaN,530010805300274,206,None,"POLYGON ((-47.7861 -15.92517, -47.7862 -15.925...",1.464300e-06,0.000484
2365035,4454,530010805300158,169.0,NaN,530010805400043,3586,None,"MULTIPOLYGON (((-47.78323 -15.92072, -47.78462...",1.014316e-10,0.000008
2365036,4454,530010805300158,169.0,NaN,530010805400082,0,None,"POLYGON ((-47.78991 -15.91907, -47.78883 -15.9...",1.449124e-04,0.000203


In [78]:
# Calcular proporção da população na área sobreposta
intersection["pop_prop"] = intersection["pop_2022"] * (intersection["area_intersection"] / intersection["area_total"])
intersection['weight'] = intersection["pop_prop"] / intersection["pop_2022"]

# Ajustar IBP considerando população
intersection["IBP_prop"] = intersection["IBP_2010"] * intersection["weight"]
intersection

,ID,CD_GEOCOD_2010,pop_2010,IBP_2010,CD_GEOCOD_2022,pop_2022,CD_FCU_2022,geometry,area_intersection,area_total,pop_prop,IBP_prop,weight
0,17182,110009812000003,555.0,3.128592,110009805000022,616,None,"MULTIPOLYGON (((-60.83109 -11.48749, -60.83352...",1.343150e-04,0.016273,5.084402e+00,0.025823,8.253899e-03
1,17182,110009812000003,555.0,3.128592,110009805000083,495,None,"POLYGON ((-60.99222 -11.5001, -60.9796 -11.483...",2.304771e-05,0.003834,2.975355e+00,0.018805,6.010819e-03
2,17182,110009812000003,555.0,3.128592,110009812000001,173,None,"MULTIPOLYGON (((-60.92784 -11.4616, -60.92784 ...",2.824522e-10,0.000033,1.461742e-03,0.000026,8.449377e-06
3,17182,110009812000003,555.0,3.128592,110009812000002,714,None,"MULTIPOLYGON (((-60.98469 -11.47198, -60.96361...",1.694423e-04,0.018971,6.377329e+00,0.027944,8.931833e-03
4,17182,110009812000003,555.0,3.128592,110009812000003,391,None,"POLYGON ((-60.89413 -11.35596, -60.89401 -11.3...",9.471774e-03,0.009793,3.781920e+02,3.026108,9.672430e-01
...,...,...,...,...,...,...,...,...,...,...,...,...,...
2365033,4454,530010805300158,169.0,NaN,530010805300273,221,None,"POLYGON ((-47.7822 -15.91286, -47.7822 -15.912...",9.990698e-15,0.000040,5.452355e-08,NaN,2.467129e-10
2365034,4454,530010805300158,169.0,NaN,530010805300274,206,None,"POLYGON ((-47.7861 -15.92517, -47.7862 -15.925...",1.464300e-06,0.000484,6.238602e-01,NaN,3.028448e-03
2365035,4454,530010805300158,169.0,NaN,530010805400043,3586,None,"MULTIPOLYGON (((-47.78323 -15.92072, -47.78462...",1.014316e-10,0.000008,4.394064e-02,NaN,1.225339e-05
2365036,4454,530010805300158,169.0,NaN,530010805400082,0,None,"POLYGON ((-47.78991 -15.91907, -47.78883 -15.9...",1.449124e-04,0.000203,0.000000e+00,NaN,NaN


In [ ]:
# Somar o IBP ajustado para cada setor de 2022
ibp_2022 = intersection.groupby("CD_GEOCOD_2022")["IBP_prop"].sum().reset_index()

# Mesclar com o shapefile de 2022
sectors_2022 = sectors_2022.merge(ibp_2022, on="CD_GEOCOD_2022", how="left")

# Salvar o novo shapefile com o IBP interpolado
sectors_2022.to_file('/Volumes/ssd/Doutorado/DATASET/IBGE/Setores Censitários/setores_censitarios_2022/br/setores_censitarios_pop_fcu_ibp.shp')

/var/folders/5l/x3ysgz0n0mvd7msxlmn75zhw0000gp/T/ipykernel_8429/2976425562.py:8: UserWarning: Column names longer than 10 characters will be truncated when saved to ESRI Shapefile.
  setores_2022.to_file('/Volumes/ssd/Doutorado/DATASET/IBGE/Setores Censitários/setores_censitarios_2022/br/setores_censitarios_pop_fcu_ibp.shp')
/Users/joaosilva/workspace/phd/classify-slum/venv/lib/python3.13/site-packages/pyogrio/raw.py:723: RuntimeWarning: Normalized/laundered field name: 'CD_GEOCOD_2022' to 'CD_GEOCOD_'
  ogr_write(
/Users/joaosilva/workspace/phd/classify-slum/venv/lib/python3.13/site-packages/pyogrio/raw.py:723: RuntimeWarning: Normalized/laundered field name: 'CD_FCU_2022' to 'CD_FCU_202'
  ogr_write(


# OP 2 - Area Interpolation

In [ ]:
# 3. Spatial Overlay
overlay = gpd.overlay(sectors_2010, sectors_2022, how='intersection')

# 4. Calculate intersection area
overlay['intersection_area'] = overlay.geometry.area

overlay

In [ ]:
# 5. Calculate total area of each 2010 sector (for normalization)
total_area_2010 = sectors_2010.geometry.area
area_dict_2010 = dict(zip(sectors_2010['CD_GEOCOD_2010'], total_area_2010))

/var/folders/5l/x3ysgz0n0mvd7msxlmn75zhw0000gp/T/ipykernel_8429/4097772198.py:2: UserWarning: Geometry is in a geographic CRS. Results from 'area' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  area_total_2010 = setores_2010.geometry.area


In [ ]:
# 6. Calculate weight of each intersection
overlay['weight'] = overlay['intersection_area'] / overlay['CD_GEOCOD_2010'].map(area_dict_2010)
overlay

,ID,CD_GEOCOD_2010,pop_2010,IBP_2010,CD_GEOCOD_2022,pop_2022,CD_FCU_2022,geometry,area_intersecao,peso
0,17182,110009812000003,555.0,3.128592,110009805000022,616,None,"MULTIPOLYGON (((-60.83109 -11.48749, -60.83352...",1.343150e-04,1.369769e-02
1,17182,110009812000003,555.0,3.128592,110009805000083,495,None,"POLYGON ((-60.99222 -11.5001, -60.9796 -11.483...",2.304771e-05,2.350449e-03
2,17182,110009812000003,555.0,3.128592,110009812000001,173,None,"MULTIPOLYGON (((-60.92784 -11.4616, -60.92784 ...",2.824522e-10,2.880500e-08
3,17182,110009812000003,555.0,3.128592,110009812000002,714,None,"MULTIPOLYGON (((-60.98469 -11.47198, -60.96361...",1.694423e-04,1.728004e-02
4,17182,110009812000003,555.0,3.128592,110009812000003,391,None,"POLYGON ((-60.89413 -11.35596, -60.89401 -11.3...",9.471774e-03,9.659493e-01
...,...,...,...,...,...,...,...,...,...,...
2179385,4454,530010805300158,169.0,NaN,530010805300273,221,None,"POLYGON ((-47.7822 -15.91286, -47.7822 -15.912...",9.990698e-15,6.469083e-11
2179386,4454,530010805300158,169.0,NaN,530010805300274,206,None,"POLYGON ((-47.7861 -15.92517, -47.7862 -15.925...",1.464300e-06,9.481498e-03
2179387,4454,530010805300158,169.0,NaN,530010805400043,3586,None,"MULTIPOLYGON (((-47.78323 -15.92072, -47.78462...",1.014316e-10,6.567801e-07
2179388,4454,530010805300158,169.0,NaN,530010805400082,0,None,"POLYGON ((-47.78991 -15.91907, -47.78883 -15.9...",1.449124e-04,9.383230e-01


In [ ]:
# 7. Calculate weighted IBP for each 2022 sector
ibp_ponderado = overlay.groupby('CD_GEOCOD_2022').apply(
    lambda x: np.sum(x['IBP_2010'] * x['weight'])
)
ibp_ponderado

/var/folders/5l/x3ysgz0n0mvd7msxlmn75zhw0000gp/T/ipykernel_8429/3029956514.py:2: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  ibp_ponderado = sobreposicao.groupby('CD_GEOCOD_2022').apply(


CD_GEOCOD_2022
110001505000002    0.008688
110001505000003    0.543099
110001505000004    1.029434
110001505000006    1.338797
110001505000007    0.397703
                     ...   
530010805440139    0.024500
530010805440140   -0.882419
530010805440141   -0.734710
530010805440142    1.330630
530010805440143    0.347390
Length: 468075, dtype: float64

In [ ]:
sectors_2022 = sectors_2022.merge(
    ibp_ponderado.reset_index(name='IBP_2010_interpolado'),
    on='CD_GEOCOD_2022',
    how='left'
)
sectors_2022

In [ ]:
sectors_2022.to_file('/Volumes/ssd/Doutorado/DATASET/IBGE/Setores Censitários/setores_censitarios_2022/br/setores_censitarios_pop_fcu_ibp_2.shp')

/var/folders/5l/x3ysgz0n0mvd7msxlmn75zhw0000gp/T/ipykernel_8429/4119778376.py:1: UserWarning: Column names longer than 10 characters will be truncated when saved to ESRI Shapefile.
  setores_2022.to_file('/Volumes/ssd/Doutorado/DATASET/IBGE/Setores Censitários/setores_censitarios_2022/br/setores_censitarios_pop_fcu_ibp_2.shp')
/Users/joaosilva/workspace/phd/classify-slum/venv/lib/python3.13/site-packages/pyogrio/raw.py:723: RuntimeWarning: Normalized/laundered field name: 'CD_GEOCOD_2022' to 'CD_GEOCOD_'
  ogr_write(
/Users/joaosilva/workspace/phd/classify-slum/venv/lib/python3.13/site-packages/pyogrio/raw.py:723: RuntimeWarning: Normalized/laundered field name: 'CD_FCU_2022' to 'CD_FCU_202'
  ogr_write(
/Users/joaosilva/workspace/phd/classify-slum/venv/lib/python3.13/site-packages/pyogrio/raw.py:723: RuntimeWarning: Normalized/laundered field name: 'IBP_2010_interpolado' to 'IBP_2010_i'
  ogr_write(


In [ ]:
overlay[overlay['CD_GEOCOD_2010'] == 110009812000003]

,ID,CD_GEOCOD_2010,pop_2010,IBP_2010,CD_GEOCOD_2022,pop_2022,CD_FCU_2022,geometry,area_intersecao,peso
0,17182,110009812000003,555.0,3.128592,110009805000022,616,None,"MULTIPOLYGON (((-60.83109 -11.48749, -60.83352...",1.343150e-04,1.369769e-02
1,17182,110009812000003,555.0,3.128592,110009805000083,495,None,"POLYGON ((-60.99222 -11.5001, -60.9796 -11.483...",2.304771e-05,2.350449e-03
2,17182,110009812000003,555.0,3.128592,110009812000001,173,None,"MULTIPOLYGON (((-60.92784 -11.4616, -60.92784 ...",2.824522e-10,2.880500e-08
3,17182,110009812000003,555.0,3.128592,110009812000002,714,None,"MULTIPOLYGON (((-60.98469 -11.47198, -60.96361...",1.694423e-04,1.728004e-02
4,17182,110009812000003,555.0,3.128592,110009812000003,391,None,"POLYGON ((-60.89413 -11.35596, -60.89401 -11.3...",9.471774e-03,9.659493e-01
5,17182,110009812000003,555.0,3.128592,110009815000002,322,None,"MULTIPOLYGON (((-60.88112 -11.39614, -60.88101...",2.369635e-08,2.416598e-06
6,17182,110009812000003,555.0,3.128592,110009815000003,273,None,"MULTIPOLYGON (((-60.89557 -11.3555, -60.8951 -...",7.060518e-06,7.200449e-04
